Created on 2026.01.27

Description: Exploratory analysis of 6 xenium samples data after CellTypist labelling. 
Goal is to first check cell labelling.

@author: bellwu

In [ ]:
# %% ---- 1.0 import libraries ----
import scanpy as sc
from pyxenium import xen_config as xc
import pandas as pd
# %% load path and directories
xen_dir = xc.xen_bwu
# load as dictionary
adata_64312 = sc.read_h5ad(xen_dir / "SCLC_64312_pp_CellTypist_labeled.h5ad")
# labeled_adatas = {p.stem: sc.read_h5ad(p) for p in xen_dir.glob("*CellTypist_labeled.h5ad")}

Want to check first sample to see how the values look. 
- Will first test on adata_64312

In [ ]:
# %% ---- 2.0 check ----
adata_64312.obs.columns
over_clustering = adata_64312.obs['over_clustering'].head(10)
pred_labels = adata_64312.obs['predicted_labels'].head(10)
maj_vote = adata_64312.obs['majority_voting'].head(10)
conf_score = adata_64312.obs['conf_score'].head(10)
print(over_clustering)
print(pred_labels)
print(maj_vote)
print(conf_score)

From values:
- overclustering gives the cluster that the cell was computationally associated with before further leiden clustering to reference
- majority voting, predicted label = gives the cell annotated type
- conf_score gives confidence voting system was

In [ ]:
adata_64312.uns

In [ ]:
adata_64312.obs.columns

In [ ]:
adata_64312.var.columns

In [ ]:
t_cells = adata_64312[adata_64312.obs['predicted_labels'].str.contains('T cell')]
percent_t_cells = t_cells.obs['predicted_labels'].value_counts() / adata_64312.n_obs * 100
percent_t_cells

Roughly 15% of all predicted cells are T cells. 
Next want to check how accurate the T cells:
- Do so based on the xenium 5k gene panel: within the panel, the curated lists contains genes that relate to T cells

In [ ]:
# load xenium 5k panel
xen_5k = pd.read_csv(xen_dir / "Xenium5k_Human_Panel.csv")
t_cell_genes = xen_5k[xen_5k['cell_type'].str.contains("T ", na=False)]

In [ ]:
# split all cell_type entries at ';' and get unique labels
all_labels = (xen_5k['cell_type']
              .dropna()
              .str.split(';')
              .explode()
              .str.strip()
              .unique())
sorted(all_labels)
# filter for all labels that do not contain T
[label for label in sorted(all_labels) if 'T' not in label]

Looking through the list, it seems that "T" does good job of selecting for only T cell-related genes.

Now need to:
1) select out the genes that are relevant to T cells from the panel.
2) see how many genes are expressed at low levels
3) create a dotplot for cells in anndata to see how the genes are expressed

In [ ]:
# select out gene names
t_cell_genes = t_cell_genes['gene_name']
type(t_cell_genes) # outputs series

In [26]:
# select out gene names as mask
mask = t_cells.var_names.isin(t_cell_genes)
t_cells = t_cells[:, mask]

In [33]:
t_cells.X[:100].toarray()

array([[0.      , 3.166625, 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       ...,
       [3.461483, 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.      ]],
      dtype=float32)